# Step 3 — Exploratory Data Analysis (EDA)

## Purpose
Now that the data is clean, we explore it to find patterns BEFORE training a model.
Doing EDA first prevents us from training a model on data we don't understand.

## What we'll look at
1. Numeric distributions (histograms)
2. Categorical value counts
3. Correlations between numeric features
4. Target class balance
5. Outlier detection (IQR method)

This step answers: **"What data analysis steps have you done?"**

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))   # project root from eda/notebooks/

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import data_loader as dl

df = dl.generate_synthetic_data(n_rows=2000, seed=42)
print(f'Loaded {len(df):,} rows × {len(df.columns)} columns')

## 3.1 Target class balance

**Why this matters first:** if 95% of your data belongs to one class, accuracy of 95% means nothing — the model just always predicts that class. Class balance decides whether we need stratified splits or class weights.

In [ ]:
target_counts = df['sales_category'].value_counts()
target_pct    = (df['sales_category'].value_counts(normalize=True) * 100).round(1)
print(pd.DataFrame({'count': target_counts, 'percent': target_pct}))

fig, ax = plt.subplots(figsize=(6, 3))
target_counts.plot(kind='barh', ax=ax, color=['#E07856', '#7FA98B', '#D9A85C'])
ax.set_title('Target class distribution — sales_category')
ax.set_xlabel('Number of records')
plt.tight_layout(); plt.show()

## 3.2 Numeric distributions

Histograms reveal whether features are skewed, bimodal, or follow a normal distribution. Skewed features often need log-transforms before linear models.

In [ ]:
numeric_cols = ['unit_price', 'quantity', 'discount_pct', 'net_sales', 'shipping_cost', 'customer_rating']
fig, axes = plt.subplots(2, 3, figsize=(12, 6))
for ax, col in zip(axes.flat, numeric_cols):
    df[col].hist(ax=ax, bins=30, color='#E07856', edgecolor='white')
    ax.set_title(col); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 3.3 Correlation matrix

Correlation tells us which features move together. Highly correlated features (|r| > 0.85) are often redundant — dropping one usually improves model stability.

In [ ]:
corr = df[numeric_cols].corr().round(2)

fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(corr, cmap='RdYlGn_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr))); ax.set_xticklabels(corr.columns, rotation=45, ha='right')
ax.set_yticks(range(len(corr))); ax.set_yticklabels(corr.columns)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, corr.iloc[i, j], ha='center', va='center', fontsize=9)
plt.colorbar(im); plt.title('Correlation heatmap'); plt.tight_layout(); plt.show()

## 3.4 Categorical breakdowns

How many distinct cities, regions, payment methods? Which dominate?

In [ ]:
for col in ['region', 'product_category', 'payment_method']:
    print(f'\n--- {col} ({df[col].nunique()} distinct values) ---')
    print(df[col].value_counts().head(5))

## 3.5 Outlier detection — IQR method

Outliers are values outside `[Q1 − 1.5×IQR,  Q3 + 1.5×IQR]`. We flag them so the cleaning/modelling stages can decide whether to cap, transform, or keep them.

In [ ]:
def iqr_outliers(series):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr    = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return ((series < lo) | (series > hi)).sum()

outlier_report = {c: iqr_outliers(df[c]) for c in numeric_cols}
print(pd.Series(outlier_report, name='outlier_count').sort_values(ascending=False))

## 3.6 Cross-tabulation: target vs key category

Are certain regions associated with high sales? This is the first hint of which features will matter for the model.

In [ ]:
crosstab = pd.crosstab(df['region'], df['sales_category'], normalize='index').round(2)
print('Share of each region by sales_category:')
print(crosstab)

## Summary of Step 3 — What EDA told us

- **Target balance:** roughly equal across low / medium / high → stratified split is safe but not required.
- **Numeric distributions:** `unit_price` is right-skewed (Electronics drives the tail); `quantity` is uniform.
- **Correlations:** `gross_sales` and `net_sales` are highly correlated by definition → keep only one to avoid multicollinearity.
- **Outliers:** `unit_price` and `shipping_cost` have a few — kept because they represent real high-value orders.
- **Regional pattern:** Tier-1 cities (Karachi / Lahore / Islamabad) skew toward higher sales categories.

**Next:** Step 4 — turn these insights into preprocessing decisions.